# NYC Property Sales: Data Import & Consolidation

This notebook downloads the NYC Property Sales dataset from Kaggle, collects and validates the individual CSV files, and consolidates them into a single PySpark DataFrame.

All file setup and pre-consolidation analysis is done in Python. All data loading and consolidation is done in PySpark.

## Imports

In [15]:
import kagglehub
import os
import pandas as pd
from collections import Counter

## Download Dataset

In [16]:
path = kagglehub.dataset_download("johnshuford/new-york-city-property-sales")

print("Path to dataset files:", path)

Path to dataset files: /Users/karinamehta/.cache/kagglehub/datasets/johnshuford/new-york-city-property-sales/versions/1


## Collect File List

Looping through the dataset directory to collect all CSV files.


**Observed issues:**
- The Bronx folder contains 4 `.ipynb_checkpoints` backup files; these would cause double counting if included.
- `BuildingClassification.csv` is a reference/lookup file, not a sales file; excluded.

**Fix:** Filter out checkpoint files and the classification reference file during collection.

In [17]:
path =  r'/Users/karinamehta/.cache/kagglehub/datasets/johnshuford/new-york-city-property-sales/versions/1'

files = []

for root,folders,filenames in os.walk(path):
    # print(root,folders,filenames)
    for f in sorted(filenames):
        if f.endswith(".csv") and f != "BuildingClassification.csv":
            files.append(os.path.join(root,f))

print(len(files))

for f in files:
    print(f.replace(path,""))

89
/StatenIsland/StatenIsland/03_StatenIsland.csv
/StatenIsland/StatenIsland/04_StatenIsland.csv
/StatenIsland/StatenIsland/05_StatenIsland.csv
/StatenIsland/StatenIsland/06_StatenIsland.csv
/StatenIsland/StatenIsland/2007_StatenIsland.csv
/StatenIsland/StatenIsland/2008_StatenIsland.csv
/StatenIsland/StatenIsland/2009_StatenIsland.csv
/StatenIsland/StatenIsland/2010_StatenIsland.csv
/StatenIsland/StatenIsland/2011_StatenIsland.csv
/StatenIsland/StatenIsland/2012_StatenIsland.csv
/StatenIsland/StatenIsland/2013_StatenIsland.csv
/StatenIsland/StatenIsland/2014_StatenIsland.csv
/StatenIsland/StatenIsland/2015_StatenIsland.csv
/StatenIsland/StatenIsland/2016_StatenIsland.csv
/StatenIsland/StatenIsland/2017_StatenIsland.csv
/StatenIsland/StatenIsland/2018_StatenIsland.csv
/StatenIsland/StatenIsland/2019_StatenIsland.csv
/Bronx/Bronx/03_Bronx.csv
/Bronx/Bronx/04_Bronx.csv
/Bronx/Bronx/05_Bronx.csv
/Bronx/Bronx/06_Bronx.csv
/Bronx/Bronx/2007_Bronx.csv
/Bronx/Bronx/2008_Bronx.csv
/Bronx/Bronx

In [18]:
#Removing the 4 files 

checkpoint_f = [f for f in files if ".ipynb_checkpoints" in f]
print(f'Removing: {checkpoint_f}')

files = [f for f in files if f not in checkpoint_f]
print(len(files))

Removing: ['/Users/karinamehta/.cache/kagglehub/datasets/johnshuford/new-york-city-property-sales/versions/1/Bronx/Bronx/.ipynb_checkpoints/2007_Bronx-checkpoint.csv', '/Users/karinamehta/.cache/kagglehub/datasets/johnshuford/new-york-city-property-sales/versions/1/Bronx/Bronx/.ipynb_checkpoints/2010_Bronx-checkpoint.csv', '/Users/karinamehta/.cache/kagglehub/datasets/johnshuford/new-york-city-property-sales/versions/1/Bronx/Bronx/.ipynb_checkpoints/2014_Bronx-checkpoint.csv', '/Users/karinamehta/.cache/kagglehub/datasets/johnshuford/new-york-city-property-sales/versions/1/Bronx/Bronx/.ipynb_checkpoints/2017_Bronx-checkpoint.csv']
85


### Verifying total count post consolidation

In [19]:
total = 0
for f in files:
    count = spark.read.csv(f, header=True).count()
    total += count
    # print(f"{os.path.basename(f)}: {count}")
print(f"\nTotal: {total}")


Total: 1626742


## Pre-Consolidation Analysis

Before merging the 85 files into one dataset, we check that they are compatible:

1. **Column names:** do all files use the same names, in the same order?
2. **Missing values:** are missing entries recorded the same way (empty vs. blank text)?

All data loading, cleaning and consolidation from this point is done in PySpark.

In [20]:
from pyspark.sql import SparkSession

#Starting SparkSession
spark = SparkSession.builder.appName("nyc-sales").getOrCreate()

#First file
old = spark.read.csv(files[0], header=True)
#Last File
new = spark.read.csv(files[-1], header=True)

print(os.path.basename(files[0]), old.columns, "\n")
print(os.path.basename(files[-1]), new.columns)

03_StatenIsland.csv ['Borough', 'Neighborhood', 'Building Class Category', 'Tax Class As Of Final Roll 18/1', 'Block', 'Lot', 'Ease-Ment', 'Building Class As Of Final Roll 18/1', 'Address', 'Apartment Number', 'Zip Code', 'Residential Units', 'Commercial Units', 'Total Units', 'Land Square Feet', 'Gross Square Feet', 'Year Built', 'Tax Class At Time Of Sale', 'Building Class At Time Of Sale', 'Sale Price', 'Sale Date'] 

2019_Queens.csv ['Borough', 'Neighborhood', 'Building Class Category', 'Tax Class At Present', 'Block', 'Lot', 'Ease-Ment', 'Building Class At Present', 'Address', 'Apartment Number', 'Zip Code', 'Residential Units', 'Commercial Units', 'Total Units', 'Land Square Feet', 'Gross Square Feet', 'Year Built', 'Tax Class At Time Of Sale', 'Building Class At Time Of Sale', 'Sale Price', 'Sale Date']


Comparing column names: oldest vs. newest file

Both `03_StatenIsland.csv` and `2019_Queens.csv` have **21 columns in the same order**.
19 of the 21 column names match exactly; **2 differ in name only**:

| Column position | 2003 file | 2019 file |
|---|---|---|
| 4 | Tax Class As Of Final Roll 18/1 | Tax Class At Present |
| 8 | Building Class As Of Final Roll 18/1 | Building Class At Present |

Next steps: validating that this is a renaming and not a structural difference.

In [21]:
# Values in the differing tax class column
old.groupBy("Tax Class As Of Final Roll 18/1").count().orderBy("count", ascending=False).show()
new.groupBy("Tax Class At Present").count().orderBy("count", ascending=False).show()

+-------------------------------+-----+
|Tax Class As Of Final Roll 18/1|count|
+-------------------------------+-----+
|                              1|10447|
|                             1A|  821|
|                              4|  443|
|                              2|  350|
|                             1B|  309|
|                               |  191|
|                             2A|   85|
|                             2B|   24|
+-------------------------------+-----+

+--------------------+-----+
|Tax Class At Present|count|
+--------------------+-----+
|                   1|16510|
|                   2| 6886|
|                   4| 1498|
|                  2A|  532|
|                  1B|  368|
|                  1A|  347|
|                  2C|  112|
|                  1D|   96|
|                  2B|   70|
|                NULL|   17|
|                  1C|    2|
+--------------------+-----+



**Note:** Missing values are recorded inconsistently. The 2003 file uses blank text (`"   "`),
while the 2019 file uses `NULL`. Spark does not treat blank text as missing, so blanks
will be converted to `NULL` during cleaning.

In [22]:
# How often does "present" tax class equal "at time of sale" tax class?
from pyspark.sql import functions as F

old.select(F.avg((F.col("Tax Class As Of Final Roll 18/1") == F.col("Tax Class At Time Of Sale")).cast("int")).alias("match_rate_old")).show()
new.select(F.avg((F.col("Tax Class At Present") == F.col("Tax Class At Time Of Sale")).cast("int")).alias("match_rate_new")).show()

+-----------------+
|   match_rate_old|
+-----------------+
|0.877663772691397|
+-----------------+

+------------------+
|    match_rate_new|
+------------------+
|0.9416751826198857|
+------------------+



**Validation:** Both columns contain the same tax class values (1, 1A, 1B, 2, 2A, 2B, 4),
and each agrees with *Tax Class At Time Of Sale* in most rows (88% in 2003, 94% in 2019).
This confirms they represent the same variable, so they will be renamed to a common name before merging.

**Next steps:** Since this comparison only covers two files, we next check the column names
of **all 85 files** to see if any other variations exist.

In [23]:
#Column names from the last file (2019 Queens), used as the reference
reference = spark.read.csv(files[-1], header=True).columns

#Sets to collect differing column names

#In some file, but not in the reference
extra_cols = set()    
#In the reference, but missing from some file
missing_cols = set()  

for f in files:
    #Read only the header row of each file to get its column names
    cols = spark.read.csv(f, header=True).columns
    #Columns this file has that the reference does not
    extra_cols.update(c for c in cols if c not in reference)
    #Columns the reference has that this file does not
    missing_cols.update(c for c in reference if c not in cols)

#If the only differences are the two columns we validated, renaming them is enough
print("Extra column names:  ", extra_cols)
print("Missing column names:", missing_cols)

Extra column names:   {'Building Class As Of Final Roll 18/1', 'Tax Class As Of Final Roll 18/1'}
Missing column names: {'Tax Class At Present', 'Building Class At Present'}


**Result:** Across all 85 files, the only column name differences are the two validated earlier
(`Tax Class` / `Building Class` *As Of Final Roll 18/1* vs. *At Present*), so renaming these
two columns is the only column-name fix needed before merging.

## Consolidation

### Preprocessing

**Observed filename issues:**
- 2003–2006 files use 2-digit years (e.g. `03_StatenIsland.csv`); 2007 onwards use 4-digit years.
- `StatenIsland` is written as one word in filenames ; needs a space for visualization.

**Fix:** Parse the year and borough from the filename, handle the 2-digit year offset, and split `StatenIsland` into `Staten Island`.

In [24]:
#Cleaning Year and names of files for consistency 
def year_and_borough(path):
    name = os.path.basename(path).replace(".csv","")
    #Splitting the basename by underscore
    year,borough = name.split("_",1)
    year = int(year)
    if year < 100:
        #Adding 2000 for the files not including 2000
        year+=2000
    if borough == "StatenIsland":
        borough = "Staten Island"
    return year, borough

#Verifying count of files within each borough
print(Counter(year_and_borough(f)[1] for f in files))
#Verifying count of files within each year
print(Counter(year_and_borough(f)[0] for f in files))

Counter({'Staten Island': 17, 'Bronx': 17, 'Manhattan': 17, 'Brooklyn': 17, 'Queens': 17})
Counter({2003: 5, 2004: 5, 2005: 5, 2006: 5, 2007: 5, 2008: 5, 2009: 5, 2010: 5, 2011: 5, 2012: 5, 2013: 5, 2014: 5, 2015: 5, 2016: 5, 2017: 5, 2018: 5, 2019: 5})


17 files per borough × 5 boroughs = 85 files. 5 files per year × 17 years (2003–2019) = 85 files. Counts look correct.

### Single File Test

Testing the full preprocessing pipeline on one file before running across all 85.

In [25]:
#Old column names -> standard names (validated earlier)
#Creating a rename map
rename_map = {
    "Tax Class As Of Final Roll 18/1": "Tax Class At Present",
    "Building Class As Of Final Roll 18/1": "Building Class At Present",
}

# Read one file, rename columns, add Year and Borough_Name from the file name
f = files[0]
year, borough = year_and_borough(f)

df = spark.read.csv(f, header=True)

for old_name, new_name in rename_map.items():
    df = df.withColumnRenamed(old_name, new_name)

#f.lit created a series within a Spark df
df = df.withColumn("Year", F.lit(year))
df = df.withColumn("Borough_Name", F.lit(borough))

print(df.columns)
df.select("Borough", "Borough_Name", "Year", "Tax Class At Present", "Sale Price").show(5)

['Borough', 'Neighborhood', 'Building Class Category', 'Tax Class At Present', 'Block', 'Lot', 'Ease-Ment', 'Building Class At Present', 'Address', 'Apartment Number', 'Zip Code', 'Residential Units', 'Commercial Units', 'Total Units', 'Land Square Feet', 'Gross Square Feet', 'Year Built', 'Tax Class At Time Of Sale', 'Building Class At Time Of Sale', 'Sale Price', 'Sale Date', 'Year', 'Borough_Name']
+-------+-------------+----+--------------------+----------+
|Borough| Borough_Name|Year|Tax Class At Present|Sale Price|
+-------+-------------+----+--------------------+----------+
|      5|Staten Island|2003|                   1|    353886|
|      5|Staten Island|2003|                   1|    310000|
|      5|Staten Island|2003|                   1|    299130|
|      5|Staten Island|2003|                   1|    304099|
|      5|Staten Island|2003|                   1|         0|
+-------+-------------+----+--------------------+----------+
only showing top 5 rows


### Full Consolidation Loop

Running across all 85 files. Using `unionByName` with `allowMissingColumns=True` to handle any schema differences safely.

In [26]:
dfs = []

for f in files:
    #Collexting year and borough from the file
    year, borough = year_and_borough(f)
    
    df = spark.read.csv(f, header=True)

    #Renaming the different column names
    for old_name, new_name in rename_map.items():
        if old_name in df.columns:
            df = df.withColumnRenamed(old_name, new_name)

    #Adding year and borough name to the df
    df = df.withColumn("Year", F.lit(year))
    df = df.withColumn("Borough_Name", F.lit(borough))

    #Adding the df to the list
    dfs.append(df)


#Initializing the combined_df with the first df within the list
combined_df = dfs[0]

for df in dfs[1:]:
    combined_df = combined_df.unionByName(df, allowMissingColumns=True)

print(f"Total rows: {combined_df.count()}")
print(f"Columns: {combined_df.columns}")
combined_df.show(5)

Total rows: 1626742
Columns: ['Borough', 'Neighborhood', 'Building Class Category', 'Tax Class At Present', 'Block', 'Lot', 'Ease-Ment', 'Building Class At Present', 'Address', 'Apartment Number', 'Zip Code', 'Residential Units', 'Commercial Units', 'Total Units', 'Land Square Feet', 'Gross Square Feet', 'Year Built', 'Tax Class At Time Of Sale', 'Building Class At Time Of Sale', 'Sale Price', 'Sale Date', 'Year', 'Borough_Name']
+-------+--------------------+-----------------------+--------------------+-----+---+---------+-------------------------+--------------------+----------------+--------+-----------------+----------------+-----------+----------------+-----------------+----------+-------------------------+------------------------------+----------+-------------------+----+-------------+
|Borough|        Neighborhood|Building Class Category|Tax Class At Present|Block|Lot|Ease-Ment|Building Class At Present|             Address|Apartment Number|Zip Code|Residential Units|Commercial 

26/10/22 16:47:48 WARN DAGScheduler: Broadcasting large task binary with size 1274.0 KiB


### Saving dataset

In [27]:
combined_df.write.mode("overwrite").parquet("data/consolidated_nyc_sales.parquet")

26/10/22 16:47:49 WARN DAGScheduler: Broadcasting large task binary with size 1263.1 KiB
26/10/22 16:47:49 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/10/22 16:47:49 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/10/22 16:47:49 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/10/22 16:47:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/10/22 16:47:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/10/22 16:47:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 

## Validating count against earlier identified number

**Result:** 1,626,742 rows × 23 columns 


## Issues to Fix in Data Cleaning

The following were observed during consolidation and will be addressed in the next notebook:

- **Blank strings vs. NULL:** Older files (pre-2007) use blank/whitespace strings for missing values; newer files use `NULL` - These need to be unified to `NULL`.
- **Whitespace in string columns:** Several string columns (e.g. `Neighborhood`, `Address`) have trailing whitespace - needs `.trim()`.
- **`Ease-Ment` column:** Appears empty across all rows - likely safe to drop.
- **Column types:** All columns are currently strings (PySpark CSV default). Numeric columns (`Sale Price`, `Gross Square Feet`, etc.) need casting.